# OpenAI Whisper — Automated Subtitle (SRT) Generator

This notebook demonstrates an automated, broadcast-ready speech-to-text pipeline using **OpenAI Whisper** — a state-of-the-art speech recognition model trained on 680,000 hours of multilingual, multitask supervised data. We transcribe an audio track and generate formatted **SubRip (.srt) subtitle files** with frame-accurate word timestamps.

| Model Variant | Parameters | VRAM Required | Relative Speed | Word Error Rate (WER) | Best Use Case |
|---|---|---|---|---|---|
| `tiny` | 39 M | ~1 GB | ~32x | Higher (~10–15%) | Ultra-fast prototyping, CPU-only edge devices |
| `base` | 74 M | ~1 GB | ~16x | Good (~7–10%) | ✅ Recommended baseline for speed & accuracy |
| `small` | 244 M | ~2 GB | ~6x | Better (~5–7%) | Specialized terminology, podcasts, interviews |
| `medium` | 769 M | ~5 GB | ~2x | High (~4–5%) | High-precision multilingual transcription |
| `large-v3` | 1550 M | ~10 GB | 1x | Frontier (<4%) | Maximum precision, noisy or overlapping audio |

> **Why Word-Level Timestamps?** Standard speech-to-text models output dense multi-sentence paragraphs. Whisper's cross-attention word timestamps allow us to segment speech into concise, broadcast-standard subtitle lines (35–42 characters, ~3–4 seconds per line) for optimal viewer retention and reading comfort.


## 1. Prerequisites & Environment Setup

### System Requirements
- **OS**: macOS 13+ (fully optimized for Apple Silicon M1/M2/M3/M4), Linux (Ubuntu 20.04+), or Windows 10/11
- **Python**: 3.10 or higher
- **FFmpeg**: Required by Whisper for audio demuxing, decoding, and resample conversion (16 kHz mono)
- **Disk Space**: ~140 MB for `base` model weights up to ~3 GB for `large-v3`, auto-cached to `~/.cache/whisper`

### Setup Instructions
1. Ensure FFmpeg is installed on your system:
   - macOS: `brew install ffmpeg`
   - Debian/Ubuntu: `sudo apt-get install ffmpeg`
2. Place your source audio (`.mp3`, `.wav`, or `.m4a`) in the working directory (default: `sample_audio.mp3`).


In [1]:
import shutil
import sys
from pathlib import Path
import torch
import whisper

# Verify core dependencies and runtime environment
print("✓ Libraries imported successfully.")
print(f"  Python executable: {sys.executable}")
print(f"  Whisper version  : {whisper.__version__}")
print(f"  PyTorch version  : {torch.__version__}")

# Hardware acceleration check (CUDA, Apple MPS, or CPU)
if torch.cuda.is_available():
    device = "cuda"
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    device_name = "Apple Silicon Metal (MPS)"
else:
    device = "cpu"
    device_name = "CPU"
print(f"  Compute device   : {device.upper()} ({device_name})")

# Verify FFmpeg binary availability
ffmpeg_bin = shutil.which("ffmpeg")
if ffmpeg_bin:
    print(f"  FFmpeg binary    : ✅ Found ({ffmpeg_bin})")
else:
    print("  FFmpeg binary    : ❌ NOT found — please install FFmpeg to process audio")


✓ Libraries imported successfully.
  Python executable: /Tutorials/poc_notebook_video/.venv/bin/python
  Whisper version  : 20250625
  PyTorch version  : 2.6.0
  Compute device   : MPS (Apple Silicon Metal (MPS))
  FFmpeg binary    : ✅ Found (/opt/homebrew/bin/ffmpeg)


## 2. Configure Input Audio & Subtitle Parameters

Define the target audio file, destination subtitle file, and typography constraints.

Industry standards for broadcast subtitles dictate:
- **`MAX_SUBTITLE_CHARS` (37–42 characters)**: Prevents multi-line overcrowding on mobile displays and video players.
- **`MAX_SUBTITLE_DURATION` (3.0–3.5 seconds)**: Ensures subtitles stay on-screen long enough to be read comfortably (~150 words per minute average reading pace).


In [2]:
# ── Configuration Parameters ──────────────────────────────────────────────────
INPUT_MP3 = Path("sample_audio.mp3")
OUTPUT_SRT = INPUT_MP3.with_suffix(".srt")

# Subtitle chunking constraints (broadcast standards)
MAX_SUBTITLE_CHARS = 42        # Max characters per subtitle line
MAX_SUBTITLE_DURATION = 3.5    # Max duration (seconds) per subtitle card

# Validate that the source audio file exists
if not INPUT_MP3.exists():
    raise FileNotFoundError(
        f"Source audio file not found: {INPUT_MP3.resolve()}\n"
        "Please place 'sample_audio.mp3' in the directory or update INPUT_MP3."
    )

print(f"Input audio  : {INPUT_MP3.resolve()}")
print(f"Output SRT   : {OUTPUT_SRT.resolve()}")
print(f"Max length   : {MAX_SUBTITLE_CHARS} characters")
print(f"Max duration : {MAX_SUBTITLE_DURATION} seconds")


Input audio  : /Users/abhisheksaxena/Tutorials/poc_notebook_video/notebook_transcribe/sample_audio.mp3
Output SRT   : /Users/abhisheksaxena/Tutorials/poc_notebook_video/notebook_transcribe/sample_audio.srt
Max length   : 42 characters
Max duration : 3.5 seconds


## 3. Listen to Input Audio

Preview the source recording directly inside the notebook to verify clarity, track duration, and speech levels before running inference.


In [3]:
from IPython.display import Audio, display

print(f"Loaded audio: {INPUT_MP3.name}")
display(Audio(str(INPUT_MP3)))


Loaded audio: sample_audio.mp3


## 4. Load the Whisper Model

Whisper weights are automatically fetched from OpenAI's repository on first run (~140 MB for `base`) and stored in `~/.cache/whisper`. Subsequent invocations load directly from the local disk cache with zero network overhead.


In [4]:
# Select model size: "tiny", "base", "small", "medium", "large-v3"
MODEL_SIZE = "base"

print(f"Loading Whisper model: '{MODEL_SIZE}' ...")
print("First run downloads weights to cache — subsequent runs load instantly.\n")

model = whisper.load_model(MODEL_SIZE)
print(f"✅ Model '{MODEL_SIZE}' loaded successfully on device: {model.device}")


Loading Whisper model: 'base' ...
First run downloads weights to cache — subsequent runs load instantly.

✅ Model 'base' loaded successfully on device: cpu


## 5. Transcribe Audio with Word-Level Timestamps

Whisper processes audio in 30-second sliding windows using an encoder-decoder Transformer architecture.

Key inference hyperparameters used below:
- **`word_timestamps=True`**: Extracts fine-grained cross-attention timestamps for every spoken word.
- **`condition_on_previous_text=False`**: Prevents the model from getting stuck in hallucination or repetition loops.
- **`temperature=0`**: Deterministic greedy decoding for maximum transcription fidelity.
- **`initial_prompt`**: Primes the model with punctuation and style context.


In [5]:
print(f"Transcribing: {INPUT_MP3.name} ...")

result = model.transcribe(
    str(INPUT_MP3),
    verbose=False,
    word_timestamps=True,
    condition_on_previous_text=False,
    compression_ratio_threshold=2.4,
    no_speech_threshold=0.6,
    logprob_threshold=-1.0,
    temperature=0,
    best_of=1,
    beam_size=1,
    initial_prompt="Generate short subtitle-friendly segments.",
)

detected_lang = result.get("language", "unknown").upper()
print(f"Detected language : {detected_lang}")
print(f"Total raw segments: {len(result['segments'])}")
print(f"\nFULL TRANSCRIPT\n{'=' * 60}")
print(result["text"].strip())


Transcribing: sample_audio.mp3 ...
Detected language : EN
Total raw segments: 1

FULL TRANSCRIPT
In today's MysteryBytes Labs session, we are learning to generate studio quality audio using Voxtral TTS on our local machine with zero subscription cost. Share your favourite voice in comments.


## 6. Parse & Format Subtitle Segments (SRT Chunking)

Raw Whisper segments can be up to 30 seconds long, which is far too long for comfortable reading. Here we implement an intelligent chunking algorithm:
1. **Timecode Formatting**: Converts raw second counts into standardized SRT timestamps (`HH:MM:SS,mmm`).
2. **Dynamic Splitting**: Groups words into readable clusters bounded by `MAX_SUBTITLE_CHARS` (42 chars) and `MAX_SUBTITLE_DURATION` (3.5s).
3. **Punctuation Clean-up**: Ensures natural punctuation spacing before rendering.


In [6]:
def format_srt_timestamp(seconds: float) -> str:
    """Convert seconds (float) to standard SRT timestamp format: HH:MM:SS,mmm"""
    total_ms = int(round(seconds * 1000))
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def split_segment_into_chunks(segment: dict, max_chars: int, max_duration: float) -> list[dict]:
    """Split a Whisper segment into shorter subtitle chunks using word timestamps."""
    words = segment.get("words") or []

    if not words:
        return [{
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"].strip(),
        }]

    chunks = []
    current_words = []
    chunk_start = None
    chunk_end = None

    for word_info in words:
        word_text = word_info["word"].strip()
        if not word_text:
            continue

        word_start = word_info.get("start", chunk_start if chunk_start is not None else segment["start"])
        word_end = word_info.get("end", word_start)

        candidate_words = current_words + [word_text]
        candidate_text = " ".join(candidate_words).strip()
        candidate_start = chunk_start if chunk_start is not None else word_start
        candidate_duration = word_end - candidate_start

        exceeds_chars = len(candidate_text) > max_chars
        exceeds_duration = candidate_duration > max_duration

        if current_words and (exceeds_chars or exceeds_duration):
            chunks.append({
                "start": chunk_start,
                "end": chunk_end,
                "text": " ".join(current_words).strip(),
            })
            current_words = [word_text]
            chunk_start = word_start
            chunk_end = word_end
        else:
            current_words = candidate_words
            chunk_start = candidate_start
            chunk_end = word_end

    if current_words:
        chunks.append({
            "start": chunk_start,
            "end": chunk_end,
            "text": " ".join(current_words).strip(),
        })

    return chunks


def build_srt_blocks(segments: list, max_chars: int, max_duration: float) -> list[str]:
    """Convert Whisper segments into standardized SRT block strings."""
    blocks = []
    subtitle_index = 1

    for segment in segments:
        for chunk in split_segment_into_chunks(segment, max_chars=max_chars, max_duration=max_duration):
            text = chunk["text"].replace(" ,", ",").replace(" .", ".").replace(" ?", "?").replace(" !", "!").strip()
            if not text:
                continue

            start = format_srt_timestamp(chunk["start"])
            end = format_srt_timestamp(chunk["end"])
            blocks.append(f"{subtitle_index}\n{start} --> {end}\n{text}")
            subtitle_index += 1

    return blocks


srt_blocks = build_srt_blocks(
    result["segments"],
    max_chars=MAX_SUBTITLE_CHARS,
    max_duration=MAX_SUBTITLE_DURATION,
)

print(f"Generated {len(srt_blocks)} formatted subtitle blocks.")
print("\nPreview (first 5 subtitle entries):\n" + "-" * 40)
for block in srt_blocks[:5]:
    print(block)
    print()


Generated 4 formatted subtitle blocks.

Preview (first 5 subtitle entries):
----------------------------------------
1
00:00:00,000 --> 00:00:03,420
In today's MysteryBytes Labs session,

2
00:00:03,420 --> 00:00:06,800
we are learning to generate studio quality

3
00:00:06,800 --> 00:00:09,100
audio using Voxtral TTS on our local machine

4
00:00:09,100 --> 00:00:10,880
with zero subscription cost.



## 7. Export & Verify SRT Subtitle File

Serialize the formatted subtitle blocks into a standard UTF-8 `.srt` file. This file can be directly imported into video editors (Premiere Pro, DaVinci Resolve, Final Cut) or media players (VLC, YouTube Studio).


In [7]:
# ── Export formatted SRT content ─────────────────────────────────────────────
srt_content = "\n\n".join(srt_blocks) + "\n"

OUTPUT_SRT.write_text(srt_content, encoding="utf-8")

file_size_kb = OUTPUT_SRT.stat().st_size / 1024
print(f"✅ SRT file exported successfully!")
print(f"   Destination      : {OUTPUT_SRT.resolve()}")
print(f"   File size        : {file_size_kb:.2f} KB")
print(f"   Total subtitles  : {len(srt_blocks)}")
print(f"   Final entry time : {srt_blocks[-1].splitlines()[1]}")


✅ SRT file exported successfully!
   Destination      : /Users/abhisheksaxena/Tutorials/poc_notebook_video/notebook_transcribe/sample_audio.srt
   File size        : 0.28 KB
   Total subtitles  : 4
   Final entry time : 00:00:09,100 --> 00:00:10,880
